In [1]:
!pip install -q openpyxl tqdm

from openpyxl import load_workbook
from tqdm.auto import tqdm
import re
import html
import unicodedata


# =========================================================
# 1. FILE PATHS
# =========================================================

MAIN_FILE = "/content/franckenachlass_7.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

LCNAF_FILE = "/content/LCNAF_updated.xlsx"
VIAF_FILE = "/content/VIAF_updated.xlsx"
WIKIDATA_PERSON_FILE = "/content/Wikidata_person_updated.xlsx"

GEONAMES_FILE = "/content/GeoNames_updated.xlsx"
WIKIDATA_PLACE_FILE = "/content/Wikidata_place_updated.xlsx"

ORGANIZATION_FILE = "/content/organization_updated.xlsx"

OUTPUT_FILE = "/content/franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

HEADER_ROW = 1


# =========================================================
# 2. MAIN FILE COLUMNS
# =========================================================

KEYWORDS_COL = "keywords"

SUBJECT_PERSON_COL = "subject_person_600"
SUBJECT_GEO_COL = "subject_geo_651"
SUBJECT_CORPORATE_COL = "subject_corporate_610"

CREATOR_LINKED_COL = "creator_linked_new"
RECEIVER_LINKED_COL = "receiver_linked_new"

SENDER_PLACE_LINKED_COL = "sender_place_linked_new"
RECEIVER_PLACE_LINKED_COL = "receiver_place_linked_new"


# =========================================================
# 3. AUTHORITY FILE COLUMNS
# =========================================================

# ---------- LCNAF PERSON ----------
LC_PERSON_LABEL = "LODLOC_label"
LC_PERSON_URI = "LODLOC_uri"

# ---------- VIAF PERSON ----------
VIAF_PERSON_LABEL = "VIAF_extractedfrWikidataentry_label"
VIAF_PERSON_URI = "VIAF_extractedfrWikidataentry_uri"

# ---------- WIKIDATA PERSON ----------
WIKI_PERSON_LABEL = "wikidata_label_final"
WIKI_PERSON_URI = "LODwikiData_uri"

# ---------- GEONAMES PLACE ----------
GEO_LABEL = "geoLabel_final"
GEO_URI = "LODgeoName_Updated"

# ---------- WIKIDATA PLACE ----------
WIKI_PLACE_LABEL = "LODwikidata_label_final"
WIKI_PLACE_URI = "LODwikiData_uri"

# ---------- ORGANIZATION ----------
ORG_MATCH_LABEL = "organizationDesc_original"

ORG_WIKI_URI = "Wikidata URL"
ORG_WIKI_LABEL = "Wikidata Label"

ORG_LC_URI = "LCNAF URL"
ORG_LC_LABEL = "LCNAF Label"

ORG_VIAF_URI = "VIAF URL"
ORG_VIAF_LABEL = "VIAF Label"


# =========================================================
# 4. GENERAL HELPERS
# =========================================================

def get_header_positions(ws):
    positions = {}

    for col in range(1, ws.max_column + 1):
        value = ws.cell(HEADER_ROW, col).value

        if value is not None:
            header = str(value).strip()
            positions.setdefault(header, []).append(col)

    return positions


def get_first_col(headers, name, required=True):
    if name in headers:
        return headers[name][0]

    if required:
        raise ValueError(f"Missing required column: {name}")

    return None


def get_last_col(headers, name, required=True):
    if name in headers:
        return headers[name][-1]

    if required:
        raise ValueError(f"Missing required column: {name}")

    return None


def is_blank(value):
    return value is None or str(value).strip() == ""


def normalize_space(value):
    if value is None:
        return ""

    return re.sub(r"\s+", " ", str(value)).strip()


def strip_html(value):
    if value is None:
        return ""

    text = re.sub(r"<[^>]+>", " ", str(value))
    text = html.unescape(text)

    return normalize_space(text)


def normalize_basic(value):
    if value is None:
        return ""

    text = strip_html(value)
    text = text.strip("[]")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def split_keywords(value):
    """
    Split keywords on semicolon/newline only.

    Do NOT split on commas because person/place names
    frequently contain commas.
    """

    if is_blank(value):
        return []

    text = str(value)

    return [
        normalize_space(part)
        for part in re.split(r"\s*;\s*|\n+", text)
        if normalize_space(part)
    ]


# =========================================================
# 5. PERSON MATCHING
# =========================================================

def strip_life_dates(name):
    if name is None:
        return ""

    name = strip_html(name)

    name = re.sub(
        r"\s*\([^)]*(?:\d{3,4}|b\.|d\.|fl\.|ca\.|c\.)[^)]*\)",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(
        r"\s*,\s*-?\d{3,4}\s*(?:[-–]\s*\d{0,4})?",
        "",
        name
    )

    name = re.sub(
        r"\b\d{3,4}\s*[-–]\s*\d{0,4}\b",
        "",
        name
    )

    name = re.sub(r"\b\d{3,4}\b", "", name)

    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r",\s*$", "", name).strip()

    return name


def person_name_keys(name):
    """
    Supports:

    Harnisch, Hans
    Hans Harnisch

    and normalized variants.
    """

    keys = set()

    clean = strip_life_dates(name)

    if not clean:
        return keys

    original_key = normalize_basic(clean)

    if original_key:
        keys.add(original_key)

    if "," in clean:
        last, first = clean.split(",", 1)

        last = normalize_space(last)
        first = normalize_space(first)

        if last and first:
            reversed_key = normalize_basic(
                f"{first} {last}"
            )

            if reversed_key:
                keys.add(reversed_key)

    else:
        parts = clean.split()

        if len(parts) >= 2:
            last = parts[-1]
            first = " ".join(parts[:-1])

            reversed_key = normalize_basic(
                f"{last}, {first}"
            )

            if reversed_key:
                keys.add(reversed_key)

    return keys


# =========================================================
# 6. PLACE MATCHING
# =========================================================

def place_keys(value):
    if value is None:
        return set()

    value = strip_html(value)

    keys = set()

    full = normalize_basic(value)

    if full:
        keys.add(("full", full))

    before_bracket = value.split("(", 1)[0].strip()

    if before_bracket:
        key = normalize_basic(before_bracket)

        if key:
            keys.add(("base", key))

    before_comma = value.split(",", 1)[0].strip()

    if before_comma:
        key = normalize_basic(before_comma)

        if key:
            keys.add(("base", key))

    return keys


# =========================================================
# 7. HTML / URI HELPERS
# =========================================================

def build_anchor(uri, label, authority):
    if is_blank(uri) or is_blank(label):
        return None

    uri = html.escape(
        str(uri).strip(),
        quote=True
    )

    label = html.escape(
        str(label).strip(),
        quote=False
    )

    return (
        f'<p><a href="{uri}">{label}</a> '
        f'[{authority}]</p>'
    )


ANY_ANCHOR_RE = re.compile(
    r'<a\b[^>]*\bhref=["\']'
    r'(?P<href>[^"\']+)'
    r'["\'][^>]*>'
    r'(?P<label>.*?)'
    r'</a>',
    re.IGNORECASE | re.DOTALL
)


def detect_authority(uri):
    text = str(uri).lower()

    if "id.loc.gov" in text:
        return "LC"

    if "viaf.org" in text:
        return "VIAF"

    if "wikidata.org" in text:
        return "Wikidata"

    if "geonames.org" in text:
        return "GeoNames"

    return None


def uri_compare_key(uri):
    """
    Ignore trailing slash when comparing duplicate URIs.
    """

    if uri is None:
        return ""

    return (
        html.unescape(str(uri))
        .strip()
        .rstrip("/")
    )


def existing_uris(cell_value):
    result = set()

    if is_blank(cell_value):
        return result

    for match in ANY_ANCHOR_RE.finditer(
        str(cell_value)
    ):
        key = uri_compare_key(
            match.group("href")
        )

        if key:
            result.add(key)

    return result


def append_anchors(cell_value, records):
    """
    Preserve existing content and add only new URIs.
    """

    existing = existing_uris(cell_value)

    output = (
        ""
        if is_blank(cell_value)
        else str(cell_value)
    )

    added = []

    for rec in records:
        uri_key = uri_compare_key(
            rec["uri"]
        )

        if not uri_key:
            continue

        if uri_key in existing:
            continue

        output += rec["anchor"]

        existing.add(uri_key)
        added.append(rec)

    return (
        output if output else None,
        added
    )


# =========================================================
# 8. LOOKUP HELPERS
# =========================================================

def add_lookup_record(lookup, keys, record):
    for key in keys:
        if not key:
            continue

        lookup.setdefault(key, [])

        existing_keys = {
            uri_compare_key(rec["uri"])
            for rec in lookup[key]
        }

        new_key = uri_compare_key(
            record["uri"]
        )

        if new_key not in existing_keys:
            lookup[key].append(record)


def unique_records(records):
    output = []
    seen = set()

    for rec in records:
        key = uri_compare_key(
            rec["uri"]
        )

        if not key or key in seen:
            continue

        seen.add(key)
        output.append(rec)

    return output


# =========================================================
# 9. LOAD PERSON AUTHORITY FILES
# =========================================================

person_lookup = {}


def load_person_authority(
    file_path,
    label_col,
    uri_col,
    authority
):

    wb = load_workbook(
        file_path,
        data_only=True
    )

    loaded = 0

    for ws in wb.worksheets:
        headers = get_header_positions(ws)

        if (
            label_col not in headers
            or uri_col not in headers
        ):
            continue

        label_idx = get_first_col(
            headers,
            label_col
        )

        uri_idx = get_first_col(
            headers,
            uri_col
        )

        print(
            f"Loading {authority} person data "
            f"from {ws.title}"
        )

        for row in range(
            HEADER_ROW + 1,
            ws.max_row + 1
        ):
            label = ws.cell(
                row,
                label_idx
            ).value

            uri = ws.cell(
                row,
                uri_idx
            ).value

            if (
                is_blank(label)
                or is_blank(uri)
            ):
                continue

            anchor = build_anchor(
                uri,
                label,
                authority
            )

            if anchor is None:
                continue

            record = {
                "category": "person",
                "authority": authority,
                "source_file": file_path,
                "source_row": row,
                "uri": str(uri).strip(),
                "label": str(label).strip(),
                "anchor": anchor
            }

            add_lookup_record(
                person_lookup,
                person_name_keys(label),
                record
            )

            loaded += 1

        break

    print(
        f"{authority} person records:",
        loaded
    )


load_person_authority(
    LCNAF_FILE,
    LC_PERSON_LABEL,
    LC_PERSON_URI,
    "LC"
)

load_person_authority(
    VIAF_FILE,
    VIAF_PERSON_LABEL,
    VIAF_PERSON_URI,
    "VIAF"
)

load_person_authority(
    WIKIDATA_PERSON_FILE,
    WIKI_PERSON_LABEL,
    WIKI_PERSON_URI,
    "Wikidata"
)


# =========================================================
# 10. LOAD PLACE AUTHORITY FILES
# =========================================================

place_full_lookup = {}
place_base_lookup = {}


def load_place_authority(
    file_path,
    label_col,
    uri_col,
    authority
):

    wb = load_workbook(
        file_path,
        data_only=True
    )

    loaded = 0

    for ws in wb.worksheets:
        headers = get_header_positions(ws)

        if (
            label_col not in headers
            or uri_col not in headers
        ):
            continue

        label_idx = get_first_col(
            headers,
            label_col
        )

        uri_idx = get_first_col(
            headers,
            uri_col
        )

        print(
            f"Loading {authority} place data "
            f"from {ws.title}"
        )

        for row in range(
            HEADER_ROW + 1,
            ws.max_row + 1
        ):
            label = ws.cell(
                row,
                label_idx
            ).value

            uri = ws.cell(
                row,
                uri_idx
            ).value

            if (
                is_blank(label)
                or is_blank(uri)
            ):
                continue

            anchor = build_anchor(
                uri,
                label,
                authority
            )

            if anchor is None:
                continue

            record = {
                "category": "place",
                "authority": authority,
                "source_file": file_path,
                "source_row": row,
                "uri": str(uri).strip(),
                "label": str(label).strip(),
                "anchor": anchor
            }

            for key_type, key in place_keys(label):

                if key_type == "full":
                    add_lookup_record(
                        place_full_lookup,
                        [key],
                        record
                    )

                elif key_type == "base":
                    add_lookup_record(
                        place_base_lookup,
                        [key],
                        record
                    )

            loaded += 1

        break

    print(
        f"{authority} place records:",
        loaded
    )


load_place_authority(
    GEONAMES_FILE,
    GEO_LABEL,
    GEO_URI,
    "GeoNames"
)

load_place_authority(
    WIKIDATA_PLACE_FILE,
    WIKI_PLACE_LABEL,
    WIKI_PLACE_URI,
    "Wikidata"
)


# =========================================================
# 11. LOAD ORGANIZATIONS
# =========================================================

organization_lookup = {}

org_wb = load_workbook(
    ORGANIZATION_FILE,
    data_only=True
)

org_ws = None
org_headers = None

for ws_temp in org_wb.worksheets:
    headers_temp = get_header_positions(
        ws_temp
    )

    if ORG_MATCH_LABEL in headers_temp:
        org_ws = ws_temp
        org_headers = headers_temp
        break


if org_ws is None:
    raise ValueError(
        "Could not find organization sheet."
    )


print(
    "Using organization sheet:",
    org_ws.title
)


org_name_idx = get_last_col(
    org_headers,
    ORG_MATCH_LABEL
)

org_wiki_uri_idx = get_first_col(
    org_headers,
    ORG_WIKI_URI,
    required=False
)

org_wiki_label_idx = get_first_col(
    org_headers,
    ORG_WIKI_LABEL,
    required=False
)

org_lc_uri_idx = get_first_col(
    org_headers,
    ORG_LC_URI,
    required=False
)

org_lc_label_idx = get_first_col(
    org_headers,
    ORG_LC_LABEL,
    required=False
)

org_viaf_uri_idx = get_first_col(
    org_headers,
    ORG_VIAF_URI,
    required=False
)

org_viaf_label_idx = get_first_col(
    org_headers,
    ORG_VIAF_LABEL,
    required=False
)


for row in range(
    HEADER_ROW + 1,
    org_ws.max_row + 1
):

    org_name = org_ws.cell(
        row,
        org_name_idx
    ).value

    if is_blank(org_name):
        continue

    key = normalize_basic(
        org_name
    )

    if not key:
        continue

    records = []

    authority_defs = [
        (
            "Wikidata",
            org_wiki_uri_idx,
            org_wiki_label_idx
        ),
        (
            "LC",
            org_lc_uri_idx,
            org_lc_label_idx
        ),
        (
            "VIAF",
            org_viaf_uri_idx,
            org_viaf_label_idx
        )
    ]

    for (
        authority,
        uri_idx,
        label_idx
    ) in authority_defs:

        if (
            uri_idx is None
            or label_idx is None
        ):
            continue

        uri = org_ws.cell(
            row,
            uri_idx
        ).value

        label = org_ws.cell(
            row,
            label_idx
        ).value

        if (
            is_blank(uri)
            or is_blank(label)
        ):
            continue

        anchor = build_anchor(
            uri,
            label,
            authority
        )

        if anchor is None:
            continue

        records.append({
            "category": "organization",
            "authority": authority,
            "source_file": ORGANIZATION_FILE,
            "source_row": row,
            "uri": str(uri).strip(),
            "label": str(label).strip(),
            "anchor": anchor
        })

    if records:
        organization_lookup.setdefault(
            key,
            []
        )

        organization_lookup[key].extend(
            records
        )


# =========================================================
# 12. AUTHORITY LOOKUPS
# =========================================================

def find_person_records(term):
    records = []

    for key in person_name_keys(term):
        records.extend(
            person_lookup.get(
                key,
                []
            )
        )

    return unique_records(
        records
    )


def find_place_records(term):
    """
    Try exact/full normalized match first.

    If not found, use base-place matching only when
    all matches represent the same normalized label.
    """

    full_key = normalize_basic(term)

    if full_key:
        exact = unique_records(
            place_full_lookup.get(
                full_key,
                []
            )
        )

        if exact:
            return exact

    base_candidates = []

    for key_type, key in place_keys(term):

        if key_type != "base":
            continue

        base_candidates.extend(
            place_base_lookup.get(
                key,
                []
            )
        )

    base_candidates = unique_records(
        base_candidates
    )

    if not base_candidates:
        return []

    normalized_labels = {
        normalize_basic(
            rec["label"]
        )
        for rec in base_candidates
    }

    if len(normalized_labels) == 1:
        return base_candidates

    return []


def find_organization_records(term):
    key = normalize_basic(term)

    if not key:
        return []

    return unique_records(
        organization_lookup.get(
            key,
            []
        )
    )


# =========================================================
# 13. FALLBACK FROM MAIN LINKED COLUMNS
# =========================================================

def extract_link_records(
    cell_value,
    category
):

    records = []

    if is_blank(cell_value):
        return records

    for match in ANY_ANCHOR_RE.finditer(
        str(cell_value)
    ):
        uri = html.unescape(
            match.group("href")
        ).strip()

        label = strip_html(
            match.group("label")
        )

        authority = detect_authority(
            uri
        )

        if authority is None:
            continue

        if category == "person":

            if authority not in [
                "LC",
                "VIAF",
                "Wikidata"
            ]:
                continue

        elif category == "place":

            if authority not in [
                "GeoNames",
                "Wikidata"
            ]:
                continue

        anchor = build_anchor(
            uri,
            label,
            authority
        )

        records.append({
            "category": category,
            "authority": authority,
            "source_file": "main linked columns",
            "source_row": None,
            "uri": uri,
            "label": label,
            "anchor": anchor
        })

    return records


def find_person_fallback(
    term,
    ws,
    row,
    creator_idx,
    receiver_idx
):

    term_keys = person_name_keys(
        term
    )

    candidates = []

    for col_idx in [
        creator_idx,
        receiver_idx
    ]:

        records = extract_link_records(
            ws.cell(
                row,
                col_idx
            ).value,
            "person"
        )

        for rec in records:

            rec_keys = person_name_keys(
                rec["label"]
            )

            if term_keys.intersection(
                rec_keys
            ):
                candidates.append(
                    rec
                )

    return unique_records(
        candidates
    )


def place_labels_compatible(
    term,
    label
):

    term_full = normalize_basic(
        term
    )

    label_full = normalize_basic(
        label
    )

    if term_full == label_full:
        return True

    term_base = {
        key
        for key_type, key
        in place_keys(term)
        if key_type == "base"
    }

    label_base = {
        key
        for key_type, key
        in place_keys(label)
        if key_type == "base"
    }

    return bool(
        term_base.intersection(
            label_base
        )
    )


def find_place_fallback(
    term,
    ws,
    row,
    sender_idx,
    receiver_idx
):

    candidates = []

    for col_idx in [
        sender_idx,
        receiver_idx
    ]:

        records = extract_link_records(
            ws.cell(
                row,
                col_idx
            ).value,
            "place"
        )

        for rec in records:

            if place_labels_compatible(
                term,
                rec["label"]
            ):
                candidates.append(
                    rec
                )

    return unique_records(
        candidates
    )


# =========================================================
# 14. LOAD MAIN WORKBOOK
# =========================================================

wb = load_workbook(
    MAIN_FILE
)

ws = None
headers = None


required_cols = [
    KEYWORDS_COL,
    SUBJECT_PERSON_COL,
    SUBJECT_GEO_COL,
    SUBJECT_CORPORATE_COL,
    CREATOR_LINKED_COL,
    RECEIVER_LINKED_COL,
    SENDER_PLACE_LINKED_COL,
    RECEIVER_PLACE_LINKED_COL
]


for sheet in wb.worksheets:

    temp_headers = get_header_positions(
        sheet
    )

    if all(
        col in temp_headers
        for col in required_cols
    ):
        ws = sheet
        headers = temp_headers
        break


if ws is None:
    raise ValueError(
        "Could not find main sheet with required columns."
    )


print(
    "Using main sheet:",
    ws.title
)


keywords_idx = get_first_col(
    headers,
    KEYWORDS_COL
)

subject_person_idx = get_first_col(
    headers,
    SUBJECT_PERSON_COL
)

subject_geo_idx = get_first_col(
    headers,
    SUBJECT_GEO_COL
)

subject_corporate_idx = get_first_col(
    headers,
    SUBJECT_CORPORATE_COL
)

creator_linked_idx = get_first_col(
    headers,
    CREATOR_LINKED_COL
)

receiver_linked_idx = get_first_col(
    headers,
    RECEIVER_LINKED_COL
)

sender_place_linked_idx = get_first_col(
    headers,
    SENDER_PLACE_LINKED_COL
)

receiver_place_linked_idx = get_first_col(
    headers,
    RECEIVER_PLACE_LINKED_COL
)


# =========================================================
# 15. PROCESS KEYWORDS
# =========================================================

match_log = []

person_terms_moved = 0
place_terms_moved = 0
organization_terms_moved = 0
unmatched_terms = 0


for row in tqdm(
    range(
        HEADER_ROW + 1,
        ws.max_row + 1
    ),
    desc="Processing keywords"
):

    original_keywords = ws.cell(
        row,
        keywords_idx
    ).value

    if is_blank(
        original_keywords
    ):
        continue


    terms = split_keywords(
        original_keywords
    )

    remaining_keywords = []


    person_value = ws.cell(
        row,
        subject_person_idx
    ).value

    geo_value = ws.cell(
        row,
        subject_geo_idx
    ).value

    corporate_value = ws.cell(
        row,
        subject_corporate_idx
    ).value


    for term in terms:

        term_used = False


        # =================================================
        # 1. PERSON TERM LISTS
        # =================================================

        person_records = find_person_records(
            term
        )

        if person_records:

            person_value, added = append_anchors(
                person_value,
                person_records
            )

            term_used = True
            person_terms_moved += 1

            match_log.append([
                row,
                term,
                "person",
                "authority term lists",
                "; ".join(
                    rec["authority"]
                    for rec in person_records
                ),
                "; ".join(
                    rec["uri"]
                    for rec in person_records
                ),
                "; ".join(
                    rec["label"]
                    for rec in person_records
                )
            ])


        # =================================================
        # 2. PLACE TERM LISTS
        # =================================================

        place_records = find_place_records(
            term
        )

        if place_records:

            geo_value, added = append_anchors(
                geo_value,
                place_records
            )

            term_used = True
            place_terms_moved += 1

            match_log.append([
                row,
                term,
                "place",
                "authority term lists",
                "; ".join(
                    rec["authority"]
                    for rec in place_records
                ),
                "; ".join(
                    rec["uri"]
                    for rec in place_records
                ),
                "; ".join(
                    rec["label"]
                    for rec in place_records
                )
            ])


        # =================================================
        # 3. ORGANIZATION TERM LIST
        # =================================================

        organization_records = find_organization_records(
            term
        )

        if organization_records:

            corporate_value, added = append_anchors(
                corporate_value,
                organization_records
            )

            term_used = True
            organization_terms_moved += 1

            match_log.append([
                row,
                term,
                "organization",
                "organization_updated",
                "; ".join(
                    rec["authority"]
                    for rec in organization_records
                ),
                "; ".join(
                    rec["uri"]
                    for rec in organization_records
                ),
                "; ".join(
                    rec["label"]
                    for rec in organization_records
                )
            ])


        # =================================================
        # IF FOUND IN ANY TERM LIST:
        # REMOVE FROM KEYWORDS
        # DO NOT NEED FALLBACK
        # =================================================

        if term_used:
            continue


        # =================================================
        # 4. PERSON FALLBACK
        # creator_linked_new / receiver_linked_new
        # =================================================

        fallback_person = find_person_fallback(
            term,
            ws,
            row,
            creator_linked_idx,
            receiver_linked_idx
        )

        if fallback_person:

            person_value, added = append_anchors(
                person_value,
                fallback_person
            )

            term_used = True
            person_terms_moved += 1

            match_log.append([
                row,
                term,
                "person",
                "creator_linked_new / receiver_linked_new",
                "; ".join(
                    rec["authority"]
                    for rec in fallback_person
                ),
                "; ".join(
                    rec["uri"]
                    for rec in fallback_person
                ),
                "; ".join(
                    rec["label"]
                    for rec in fallback_person
                )
            ])


        # =================================================
        # 5. PLACE FALLBACK
        # sender_place_linked_new / receiver_place_linked_new
        # =================================================

        fallback_place = find_place_fallback(
            term,
            ws,
            row,
            sender_place_linked_idx,
            receiver_place_linked_idx
        )

        if fallback_place:

            geo_value, added = append_anchors(
                geo_value,
                fallback_place
            )

            term_used = True
            place_terms_moved += 1

            match_log.append([
                row,
                term,
                "place",
                "sender_place_linked_new / receiver_place_linked_new",
                "; ".join(
                    rec["authority"]
                    for rec in fallback_place
                ),
                "; ".join(
                    rec["uri"]
                    for rec in fallback_place
                ),
                "; ".join(
                    rec["label"]
                    for rec in fallback_place
                )
            ])


        # =================================================
        # IF FALLBACK USED:
        # REMOVE FROM KEYWORDS
        # =================================================

        if term_used:
            continue


        # =================================================
        # NOTHING FOUND
        # KEEP EXACT TERM IN KEYWORDS
        # =================================================

        remaining_keywords.append(
            term
        )

        unmatched_terms += 1

        match_log.append([
            row,
            term,
            "unmatched",
            "kept in keywords",
            "",
            "",
            ""
        ])


    # =====================================================
    # WRITE UPDATED CELLS
    # =====================================================

    ws.cell(
        row,
        subject_person_idx
    ).value = person_value

    ws.cell(
        row,
        subject_geo_idx
    ).value = geo_value

    ws.cell(
        row,
        subject_corporate_idx
    ).value = corporate_value


    ws.cell(
        row,
        keywords_idx
    ).value = (
        "; ".join(
            remaining_keywords
        )
        if remaining_keywords
        else None
    )


# =========================================================
# 16. CREATE LOG SHEET
# =========================================================

LOG_SHEET = "keyword_authority_log"


if LOG_SHEET in wb.sheetnames:
    del wb[
        LOG_SHEET
    ]


log_ws = wb.create_sheet(
    LOG_SHEET
)


log_ws.append([
    "export_row",
    "keyword_term",
    "result_category",
    "source_used",
    "authorities",
    "uris",
    "labels"
])


for item in match_log:
    log_ws.append(
        item
    )


for col in range(
    1,
    log_ws.max_column + 1
):
    log_ws.column_dimensions[
        log_ws.cell(
            1,
            col
        ).column_letter
    ].width = 40


# =========================================================
# 17. SAVE SAFELY
# =========================================================

for sheet in wb.worksheets:
    sheet.sheet_state = "visible"


wb.active = wb.sheetnames.index(
    ws.title
)


if wb.views:

    wb.views[0].activeTab = (
        wb.sheetnames.index(
            ws.title
        )
    )

    wb.views[0].firstSheet = 0


wb.save(
    OUTPUT_FILE
)


# =========================================================
# 18. SUMMARY
# =========================================================

print()
print("Done.")
print("Saved file:", OUTPUT_FILE)
print()

print(
    "Person terms used:",
    person_terms_moved
)

print(
    "Place terms used:",
    place_terms_moved
)

print(
    "Organization terms used:",
    organization_terms_moved
)

print(
    "Unmatched terms left in keywords:",
    unmatched_terms
)

Loading LC person data from Sheet1
LC person records: 238
Loading VIAF person data from Sheet1
VIAF person records: 324
Loading Wikidata person data from Sheet1
Wikidata person records: 847
Loading GeoNames place data from Sheet1
GeoNames place records: 466
Wikidata place records: 0
Using organization sheet: Sheet1
Using main sheet: Sheet1


Processing keywords:   0%|          | 0/324 [00:00<?, ?it/s]


Done.
Saved file: /content/franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx

Person terms used: 0
Place terms used: 0
Organization terms used: 0
Unmatched terms left in keywords: 3
